In [ ]:
# ============================================================
#  Sentiment Classification using LSTM (Gutenberg + NLTK)
# ============================================================

# Step 1: Import libraries
import nltk
import nltk
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('vader_lexicon')
nltk.download('stopwords')


from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Step 2: Load Hamlet text
text = gutenberg.raw('shakespeare-hamlet.txt')
print("Total characters:", len(text))

# Step 3: Sentence tokenization
sentences = sent_tokenize(text)
print("Total sentences:", len(sentences))

# Step 4: Sentiment labeling using VADER
sia = SentimentIntensityAnalyzer()
data = []
for sent in sentences:
    score = sia.polarity_scores(sent)['compound']
    if score >= 0.05:
        label = 1   # Positive
    elif score <= -0.05:
        label = 0   # Negative
    else:
        continue    # Skip neutral sentences
    data.append((sent, label))

df = pd.DataFrame(data, columns=['sentence', 'label'])
print("Dataset size after labeling:", len(df))
print(df.head())

# Step 5: Text Preprocessing
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = [w for w in word_tokenize(text) if w not in stop_words]
    return ' '.join(words)

df['clean_sentence'] = df['sentence'].apply(clean_text)

# Step 6: Tokenization and padding
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_sentence'])
sequences = tokenizer.texts_to_sequences(df['clean_sentence'])
padded = pad_sequences(sequences, maxlen=30, padding='post', truncating='post')

X = np.array(padded)
y = np.array(df['label'])

# Step 7: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Build LSTM model
model = Sequential([
    Embedding(10000, 128, input_length=30),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer=Adam(1e-3), metrics=['accuracy'])
model.summary()

# Step 9: Train the model
history = model.fit(X_train, y_train, validation_split=0.2, epochs=5, batch_size=32, verbose=1)

# Step 10: Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Accuracy: {accuracy*100:.2f}%")

# Step 11: Example predictions
sample_texts = [
    "I love thee more than words can wield the matter.",
    "O wretched state! O bosom black as death!"
]
sample_clean = [clean_text(t) for t in sample_texts]
seq = tokenizer.texts_to_sequences(sample_clean)
pad = pad_sequences(seq, maxlen=30, padding='post')
pred = model.predict(pad)

for t, p in zip(sample_texts, pred):
    sentiment = "Positive" if p > 0.5 else "Negative"
    print(f"{t} → {sentiment} ({p[0]:.2f})")


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Total characters: 162881
Total sentences: 2355
Dataset size after labeling: 923
                                            sentence  label
0  You come most carefully vpon your houre\n\n   ...      1
1  For this releefe much thankes: 'Tis bitter col...      0
2                                   Well, goodnight.      1
3                  Friends to this ground\n\n   Mar.      1
4                     Giue you good night\n\n   Mar.      1


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 9s 128ms/step - accuracy: 0.6184 - loss: 0.6780 - val_accuracy: 0.5811 - val_loss: 0.6821
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.6167 - loss: 0.6722 - val_accuracy: 0.5811 - val_loss: 0.6835
Epoch 3/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.6087 - loss: 0.6658 - val_accuracy: 0.5811 - val_loss: 0.6806
Epoch 4/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.6311 - loss: 0.6318 - val_accuracy: 0.6622 - val_loss: 0.9344
Epoch 5/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.8787 - loss: 0.3802 - val_accuracy: 0.6959 - val_loss: 0.7826
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7086 - loss: 0.7908

Test Accuracy: 72.97%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step
I love thee more than words can wield the matter. → Negative (0.05)
O wretched state! O bosom black as death! → Negative (0.06)
